# Notebook 01 — Ingestion API → MongoDB et simulation des paiements
## Projet fil rouge : BigData Commerce Analytics · Jour 2 · B2 Sup de Vinci

**Objectif :** charger les 8 ressources DummyJSON et le fichier des commandes dans la base MongoDB `commerce`, puis simuler un paiement par panier.

**Prérequis :** MongoDB démarré avec Docker (`docker start mongo`), port 27017.

> Les données DummyJSON sont fictives. Les paiements sont **simulés** : ils ne correspondent à aucune transaction réelle.

## 0. Connexion à MongoDB

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, random
import requests
import pandas as pd
from pymongo import MongoClient

# Dossier du projet (le notebook est dans notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
CSV_COMMANDES = ROOT / "data" / "processed" / "commandes_ecommerce.csv"
BASE = "https://dummyjson.com"

client = MongoClient("mongodb://localhost:27017")
print("Le serveur répond :", client.admin.command("ping"))
db = client["commerce"]
print("Base sélectionnée :", db.name)

Le serveur répond : {'ok': 1.0}
Base sélectionnée : commerce


## Atelier 1 (1/2) — Charger les 8 ressources API dans MongoDB

Pour chaque ressource :
1. on appelle l'API avec `limit=0` (tous les éléments) ; si l'API ne répond pas, on reprend le fichier JSON collecté au jour 1 dans `data/raw/` ;
2. on **vide** la collection avant d'insérer, pour pouvoir relancer la cellule sans créer de doublons ;
3. on vérifie que le nombre de documents insérés est égal au `total` annoncé par l'API.

In [2]:
resources = ["products", "carts", "users", "posts", "comments", "recipes", "quotes", "todos"]

def charger_ressource(resource):
    try:
        r = requests.get(f"{BASE}/{resource}?limit=0", timeout=15)
        r.raise_for_status()
        return r.json(), "API"
    except Exception:
        return json.loads((RAW / f"{resource}.json").read_text(encoding="utf-8")), "fichier local"

date_ingestion = datetime.now(timezone.utc).isoformat(timespec="seconds")
bilan = []
for resource in resources:
    payload, origine = charger_ressource(resource)
    documents = payload[resource]
    collection = db[f"raw_{resource}"]
    collection.delete_many({})          # on repart d'une collection vide
    collection.insert_many(documents)
    nb_mongo = collection.count_documents({})
    bilan.append({"ressource": resource, "collection": f"raw_{resource}", "origine": origine,
                  "total_api": payload.get("total"), "documents_mongo": nb_mongo,
                  "controle": "OK" if nb_mongo == payload.get("total") else "ÉCART"})

print("Ingestion effectuée le :", date_ingestion)
display(pd.DataFrame(bilan))

Ingestion effectuée le : 2026-11-27T09:31:08+00:00


,ressource,collection,origine,total_api,documents_mongo,controle
0,products,raw_products,API,194,194,OK
1,carts,raw_carts,API,208,208,OK
2,users,raw_users,API,208,208,OK
3,posts,raw_posts,API,251,251,OK
4,comments,raw_comments,API,340,340,OK
5,recipes,raw_recipes,API,50,50,OK
6,quotes,raw_quotes,API,1454,1454,OK
7,todos,raw_todos,API,254,254,OK


In [12]:
# Exemple de document stocké : MongoDB ajoute automatiquement un champ technique _id
db["raw_carts"].find_one({}, {"_id": 1, "id": 1, "userId": 1, "total": 1, "discountedTotal": 1, "totalProducts": 1})

{'_id': ObjectId('6b080b199e77dc3805bfd3b4'),
 'id': 1,
 'total': 13037.88,
 'discountedTotal': 11510.81,
 'userId': 1,
 'totalProducts': 4}

## Atelier 1 (2/2) — 1. Importer les commandes dans `raw_commandes`

Fichier : `data/processed/commandes_ecommerce.csv` (une ligne = un produit dans un panier).

Les valeurs manquantes (`NaN`, colonne `brand`) sont converties en `None` pour être stockées comme `null` dans MongoDB, sans les remplacer par un faux texte.

In [13]:
df = pd.read_csv(CSV_COMMANDES)
print("CSV :", df.shape[0], "lignes,", df.shape[1], "colonnes,", df["cart_id"].nunique(), "paniers")

# NaN -> None pour que MongoDB stocke null
records = df.astype(object).where(df.notna(), None).to_dict("records")

col_commandes = db["raw_commandes"]
col_commandes.delete_many({})
resultat = col_commandes.insert_many(records)
print("Documents insérés :", len(resultat.inserted_ids))

CSV : 800 lignes, 16 colonnes, 208 paniers
Documents insérés : 800


In [14]:
# Contrôles d'intégrité et de volume
controles_import = {
    "lignes_csv": len(df),
    "documents_mongo": col_commandes.count_documents({}),
    "paniers_distincts_mongo": len(col_commandes.distinct("cart_id")),
    "brand_null_mongo": col_commandes.count_documents({"brand": None}),
    "brand_manquant_csv": int(df["brand"].isna().sum()),
    "somme_line_total_csv": round(df["line_total"].sum(), 2),
    "somme_line_total_mongo": round(list(col_commandes.aggregate([{"$group": {"_id": None, "s": {"$sum": "$line_total"}}}]))[0]["s"], 2),
}
display(pd.Series(controles_import).to_frame("valeur"))
assert controles_import["lignes_csv"] == controles_import["documents_mongo"], "Nombre de documents différent du CSV"
print("Import contrôlé : volume et montants identiques au CSV.")

,valeur
lignes_csv,800.00
documents_mongo,800.00
paniers_distincts_mongo,208.00
brand_null_mongo,377.00
brand_manquant_csv,377.00
somme_line_total_csv,3834278.63
somme_line_total_mongo,3834278.63


Import contrôlé : volume et montants identiques au CSV.


## Atelier 1 (2/2) — 2. Simuler les paiements dans `raw_payments`

**Processus :**
1. On regroupe les lignes de commande par `cart_id` pour obtenir **un seul enregistrement par panier** (208 paniers).
2. Pour chaque panier, on reprend : `cart_id`, `order_id` (la commande), `user_id` (le client) et `cart_discounted_total` (le montant à payer, après remise). La date du paiement est l'`order_date` de la première ligne du panier (date **synthétique**). On ne prend pas la plus petite date : comme les dates sont tirées au hasard ligne par ligne, le minimum tomberait presque toujours en début d'année.
3. On tire un statut au hasard avec `random.choices` et les poids 70 / 20 / 10. La graine fixe (`2026`) permet de retrouver les mêmes résultats à chaque exécution.
4. On crée un **index unique** sur `cart_id` : MongoDB refusera toute deuxième tentative de paiement pour un même panier.

In [15]:
paniers = (df.groupby("cart_id")
             .agg(order_id=("order_id", "first"),
                  user_id=("user_id", "first"),
                  amount=("cart_discounted_total", "first"),
                  payment_date=("order_date", "first"))
             .reset_index())
print("Paniers distincts :", len(paniers))

rng = random.Random(2026)   # graine fixe : simulation reproductible
statuts = rng.choices(["PAID", "REFUSED", "PENDING"], weights=[70, 20, 10], k=len(paniers))

payments = []
for i, (panier, statut) in enumerate(zip(paniers.to_dict("records"), statuts), start=1):
    payments.append({
        "payment_id": f"PAY-{i:05d}",
        "cart_id": int(panier["cart_id"]),
        "order_id": panier["order_id"],
        "user_id": int(panier["user_id"]),
        "amount": float(panier["amount"]),
        "status": statut,
        "payment_date": panier["payment_date"],
        "simulated": True,
    })

col_payments = db["raw_payments"]
col_payments.drop()
col_payments.create_index("cart_id", unique=True)   # un seul paiement par panier
col_payments.insert_many(payments)
print("Paiements insérés :", col_payments.count_documents({}))

Paniers distincts : 208
Paiements insérés : 208


In [16]:
# Contrôles de cohérence des paiements simulés
nb_paiements = col_payments.count_documents({})
nb_paniers_payes = len(col_payments.distinct("cart_id"))
statuts_trouves = sorted(col_payments.distinct("status"))
somme_paiements = round(sum(p["amount"] for p in col_payments.find({}, {"amount": 1})), 2)
somme_paniers = round(paniers["amount"].sum(), 2)

display(pd.Series({
    "paiements": nb_paiements,
    "paniers_distincts_dans_payments": nb_paniers_payes,
    "paniers_distincts_dans_commandes": len(col_commandes.distinct("cart_id")),
    "statuts_trouvés": ", ".join(statuts_trouves),
    "montants_négatifs": col_payments.count_documents({"amount": {"$lt": 0}}),
    "somme_montants_paiements": somme_paiements,
    "somme_montants_paniers": somme_paniers,
}).to_frame("valeur"))

assert nb_paiements == nb_paniers_payes == 208, "Il doit y avoir exactement 1 paiement par panier"
assert statuts_trouves == ["PAID", "PENDING", "REFUSED"]
assert somme_paiements == somme_paniers
print("Contrôles OK : 208 paiements, 1 par panier, montants cohérents.")

,valeur
paiements,208
paniers_distincts_dans_payments,208
paniers_distincts_dans_commandes,208
statuts_trouvés,"PAID, PENDING, REFUSED"
montants_négatifs,0
somme_montants_paiements,3456709.58
somme_montants_paniers,3456709.58


Contrôles OK : 208 paiements, 1 par panier, montants cohérents.


## Atelier 1 (2/2) — 3. Analyser les résultats

In [17]:
resultats = list(col_payments.aggregate([
    {"$group": {"_id": "$status", "nombre": {"$sum": 1}, "montant_total": {"$sum": "$amount"}}},
    {"$sort": {"nombre": -1}},
]))
analyse = pd.DataFrame(resultats).rename(columns={"_id": "statut"})[["statut", "nombre", "montant_total"]]
analyse["part_%"] = (analyse["nombre"] / analyse["nombre"].sum() * 100).round(1)
analyse["montant_total"] = analyse["montant_total"].round(2)
display(analyse)

encaisse = analyse.loc[analyse["statut"] == "PAID", "montant_total"].sum()
print(f"Montant réellement encaissé (PAID uniquement) : {encaisse:,.2f}")

,statut,nombre,montant_total,part_%
0,PAID,145,2322729.73,69.7
1,REFUSED,42,915022.83,20.2
2,PENDING,21,218957.02,10.1


Montant réellement encaissé (PAID uniquement) : 2,322,729.73


### Réponses aux questions du TP

**Quel processus génère une tentative de paiement unique par panier ?**
Un regroupement (`groupby`) des lignes de commande par `cart_id` : on passe de 800 lignes produit à 208 paniers, puis on crée un paiement par panier. L'index unique sur `cart_id` dans MongoDB empêche ensuite tout doublon.

**Comment reproduire les trois issues en respectant la répartition 70 / 20 / 10 ?**
Avec un tirage aléatoire pondéré (`random.choices` avec les poids 70, 20 et 10). Sur 208 paniers, les proportions observées sont proches des probabilités, mais pas exactement égales. La graine fixe rend le résultat reproductible.

**Quels éléments du panier reprendre ?**
`cart_id` (le panier), `order_id` (la commande), `user_id` (le client) et `cart_discounted_total` (le montant à payer après remise). On ne prend pas `cart_total`, qui est le montant avant remise.

**Quels contrôles vérifient la cohérence et l'insertion ?**
Nombre de paiements = nombre de paniers distincts (208) ; un `cart_id` n'apparaît qu'une fois (index unique) ; seuls les statuts PAID, REFUSED et PENDING existent ; aucun montant négatif ; la somme des paiements est égale à la somme des montants des paniers.

**Comment s'assurer qu'un panier n'a pas été comptabilisé plusieurs fois ?**
Le nombre de `cart_id` distincts dans `raw_payments` est égal au nombre de documents, et l'index unique rejette toute deuxième insertion. On calcule aussi les montants à partir de `raw_payments` (1 ligne par panier) et non en additionnant `cart_total` sur les lignes produit, où il est répété.

**Règle métier :** seuls les paiements PAID sont encaissés. REFUSED et PENDING ne comptent pas dans le chiffre d'affaires encaissé.

In [18]:
# Récapitulatif des collections de la base commerce
display(pd.DataFrame([{"collection": c, "documents": db[c].count_documents({})} for c in sorted(db.list_collection_names())]))

,collection,documents
0,raw_carts,208
1,raw_commandes,800
2,raw_comments,340
3,raw_payments,208
4,raw_posts,251
5,raw_products,194
6,raw_quotes,1454
7,raw_recipes,50
8,raw_todos,254
9,raw_users,208
